In [73]:
# %pip install xgboost
# %pip install graphviz --upgrade
# %pip install lightgbm

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [3]:
ross_df = pd.read_csv(r'D:\Codes\Machine_Learning\Scikit_Learn\Learning\train.csv', dtype={'StateHoliday':'string'}).copy()
store_df = pd.read_csv(r'D:\Codes\Machine_Learning\Scikit_Learn\Learning\store.csv').copy()
test_df = pd.read_csv(r'D:\Codes\Machine_Learning\Scikit_Learn\Learning\test.csv').copy()

In [76]:
merged_df = ross_df.merge(store_df, how='left', on='Store')
merged_test_df = test_df.merge(store_df, how='left', on='Store')

In [77]:
merged_df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval'],
      dtype='object')

In [78]:
merged_df.isna().sum()

Store                             0
DayOfWeek                         0
Date                              0
Sales                             0
Customers                         0
Open                              0
Promo                             0
StateHoliday                      0
SchoolHoliday                     0
StoreType                         0
Assortment                        0
CompetitionDistance            2642
CompetitionOpenSinceMonth    323348
CompetitionOpenSinceYear     323348
Promo2                            0
Promo2SinceWeek              508031
Promo2SinceYear              508031
PromoInterval                508031
dtype: int64

In [79]:
merged_df

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval
0,1,5,2015-07-31,5263,555,1,1,0,1,c,a,1270.0,9.0,2008.0,0,NaN,NaN,NaN
1,2,5,2015-07-31,6064,625,1,1,0,1,a,a,570.0,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct"
2,3,5,2015-07-31,8314,821,1,1,0,1,a,a,14130.0,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct"
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,c,620.0,9.0,2009.0,0,NaN,NaN,NaN
4,5,5,2015-07-31,4822,559,1,1,0,1,a,a,29910.0,4.0,2015.0,0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1017204,1111,2,2013-01-01,0,0,0,0,a,1,a,a,1900.0,6.0,2014.0,1,31.0,2013.0,"Jan,Apr,Jul,Oct"
1017205,1112,2,2013-01-01,0,0,0,0,a,1,c,c,1880.0,4.0,2006.0,0,NaN,NaN,NaN
1017206,1113,2,2013-01-01,0,0,0,0,a,1,a,c,9260.0,NaN,NaN,0,NaN,NaN,NaN
1017207,1114,2,2013-01-01,0,0,0,0,a,1,a,c,870.0,NaN,NaN,0,NaN,NaN,NaN


In [80]:
merged_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1017209 entries, 0 to 1017208
Data columns (total 18 columns):
 #   Column                     Non-Null Count    Dtype  
---  ------                     --------------    -----  
 0   Store                      1017209 non-null  int64  
 1   DayOfWeek                  1017209 non-null  int64  
 2   Date                       1017209 non-null  object 
 3   Sales                      1017209 non-null  int64  
 4   Customers                  1017209 non-null  int64  
 5   Open                       1017209 non-null  int64  
 6   Promo                      1017209 non-null  int64  
 7   StateHoliday               1017209 non-null  string 
 8   SchoolHoliday              1017209 non-null  int64  
 9   StoreType                  1017209 non-null  object 
 10  Assortment                 1017209 non-null  object 
 11  CompetitionDistance        1014567 non-null  float64
 12  CompetitionOpenSinceMonth  693861 non-null   float64
 13  CompetitionO

In [81]:
def split_date(df):
    df['Date'] = pd.to_datetime(df['Date'])
    df['Year'] = df.Date.dt.year
    df['Month'] = df.Date.dt.month
    df['Day'] = df.Date.dt.day
    df['WeekOfYear'] = df.Date.dt.isocalendar().week

In [82]:
split_date(merged_df)
split_date(merged_test_df)

In [83]:
merged_test_df.head(10)

,Id,Store,DayOfWeek,Date,Open,Promo,StateHoliday,SchoolHoliday,StoreType,Assortment,...,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,WeekOfYear
0,1,1,4,2015-09-17,1.0,1,0,0,c,a,...,9.0,2008.0,0,NaN,NaN,NaN,2015,9,17,38
1,2,3,4,2015-09-17,1.0,1,0,0,a,a,...,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,9,17,38
2,3,7,4,2015-09-17,1.0,1,0,0,a,c,...,4.0,2013.0,0,NaN,NaN,NaN,2015,9,17,38
3,4,8,4,2015-09-17,1.0,1,0,0,a,a,...,10.0,2014.0,0,NaN,NaN,NaN,2015,9,17,38
4,5,9,4,2015-09-17,1.0,1,0,0,a,c,...,8.0,2000.0,0,NaN,NaN,NaN,2015,9,17,38
5,6,10,4,2015-09-17,1.0,1,0,0,a,a,...,9.0,2009.0,0,NaN,NaN,NaN,2015,9,17,38
6,7,11,4,2015-09-17,1.0,1,0,0,a,c,...,11.0,2011.0,1,1.0,2012.0,"Jan,Apr,Jul,Oct",2015,9,17,38
7,8,12,4,2015-09-17,1.0,1,0,0,a,c,...,NaN,NaN,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,9,17,38
8,9,13,4,2015-09-17,1.0,1,0,0,d,a,...,NaN,NaN,1,45.0,2009.0,"Feb,May,Aug,Nov",2015,9,17,38
9,10,14,4,2015-09-17,1.0,1,0,0,a,a,...,3.0,2014.0,1,40.0,2011.0,"Jan,Apr,Jul,Oct",2015,9,17,38


In [84]:
merged_df[merged_df.Open == 0 ].Sales.value_counts()

Sales
0    172817
Name: count, dtype: int64

In [85]:
merged_df = merged_df[merged_df['Open'] == 1].copy()
merged_df.head()

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,CompetitionOpenSinceMonth,CompetitionOpenSinceYear,Promo2,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,WeekOfYear
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,9.0,2008.0,0,NaN,NaN,NaN,2015,7,31,31
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,11.0,2007.0,1,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,31
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,12.0,2006.0,1,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,31
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,...,9.0,2009.0,0,NaN,NaN,NaN,2015,7,31,31
4,5,5,2015-07-31,4822,559,1,1,0,1,a,...,4.0,2015.0,0,NaN,NaN,NaN,2015,7,31,31


In [86]:
merged_df['Open'].value_counts()

Open
1    844392
Name: count, dtype: int64

## Competition

We can use the columns `CompetitionOpenSince[Month/Year]` column from store_df to compute the number of months for which a competitor has been open near the store.

In [87]:
def comp_months(df):
    df['CompetitionOpen'] = 12 * (df.Year - df.CompetitionOpenSinceYear) + (df.Month - df.CompetitionOpenSinceMonth)
    #to tackle the future entries
    df['CompetitionOpen'] = df['CompetitionOpen'].map(lambda x: 0 if x < 0 else x).fillna(0)

In [88]:
comp_months(merged_df)
comp_months(merged_test_df)

In [89]:
merged_df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day',
       'WeekOfYear', 'CompetitionOpen'],
      dtype='object')

In [90]:
merged_df[['Date', 'CompetitionDistance', 'CompetitionOpenSinceYear', 'CompetitionOpenSinceMonth', 'CompetitionOpen']]

,Date,CompetitionDistance,CompetitionOpenSinceYear,CompetitionOpenSinceMonth,CompetitionOpen
0,2015-07-31,1270.0,2008.0,9.0,82.0
1,2015-07-31,570.0,2007.0,11.0,92.0
2,2015-07-31,14130.0,2006.0,12.0,103.0
3,2015-07-31,620.0,2009.0,9.0,70.0
4,2015-07-31,29910.0,2015.0,4.0,3.0
...,...,...,...,...,...
1016776,2013-01-01,150.0,2006.0,9.0,76.0
1016827,2013-01-01,860.0,1999.0,10.0,159.0
1016863,2013-01-01,840.0,NaN,NaN,0.0
1017042,2013-01-01,1430.0,NaN,NaN,0.0


Adding some additional columns to indicate how long a store has been running `Promo2` or whether a new round of `Promo2` starts in the current month.

In [91]:
def check_promo_month(row):
    month2str = {1:'Jan', 2:'Feb', 3:'Mar', 4:'Apr', 5:'May', 6:'Jun',              
                 7:'Jul', 8:'Aug', 9:'Sept', 10:'Oct', 11:'Nov', 12:'Dec'}
    try:
        months = (row['PromoInterval'] or '').split(',')
        if row['Promo2Open'] and month2str[row['Month']] in months:
            return 1
        else:
            return 0
    except Exception:
        return 0

def promo_cols(df):
    # Months since Promo2 was open
    df['Promo2Open'] = 12 * (df.Year - df.Promo2SinceYear) +  (df.WeekOfYear - df.Promo2SinceWeek)*7/30.5
    df['Promo2Open'] = df['Promo2Open'].map(lambda x: 0 if x < 0 else x).fillna(0) * df['Promo2']
    # Whether a new round of promotions was started in the current month
    df['IsPromo2Month'] = df.apply(check_promo_month, axis=1) * df['Promo2']

In [92]:
promo_cols(merged_df)
promo_cols(merged_test_df)

In [93]:
merged_df.head(10)

,Store,DayOfWeek,Date,Sales,Customers,Open,Promo,StateHoliday,SchoolHoliday,StoreType,...,Promo2SinceWeek,Promo2SinceYear,PromoInterval,Year,Month,Day,WeekOfYear,CompetitionOpen,Promo2Open,IsPromo2Month
0,1,5,2015-07-31,5263,555,1,1,0,1,c,...,NaN,NaN,NaN,2015,7,31,31,82.0,0.000000,0
1,2,5,2015-07-31,6064,625,1,1,0,1,a,...,13.0,2010.0,"Jan,Apr,Jul,Oct",2015,7,31,31,92.0,64.131148,1
2,3,5,2015-07-31,8314,821,1,1,0,1,a,...,14.0,2011.0,"Jan,Apr,Jul,Oct",2015,7,31,31,103.0,51.901639,1
3,4,5,2015-07-31,13995,1498,1,1,0,1,c,...,NaN,NaN,NaN,2015,7,31,31,70.0,0.000000,0
4,5,5,2015-07-31,4822,559,1,1,0,1,a,...,NaN,NaN,NaN,2015,7,31,31,3.0,0.000000,0
5,6,5,2015-07-31,5651,589,1,1,0,1,a,...,NaN,NaN,NaN,2015,7,31,31,19.0,0.000000,0
6,7,5,2015-07-31,15344,1414,1,1,0,1,a,...,NaN,NaN,NaN,2015,7,31,31,27.0,0.000000,0
7,8,5,2015-07-31,8492,833,1,1,0,1,a,...,NaN,NaN,NaN,2015,7,31,31,9.0,0.000000,0
8,9,5,2015-07-31,8565,687,1,1,0,1,a,...,NaN,NaN,NaN,2015,7,31,31,179.0,0.000000,0
9,10,5,2015-07-31,7185,681,1,1,0,1,a,...,NaN,NaN,NaN,2015,7,31,31,70.0,0.000000,0


## Input and Target Columns

In [94]:
merged_df.columns

Index(['Store', 'DayOfWeek', 'Date', 'Sales', 'Customers', 'Open', 'Promo',
       'StateHoliday', 'SchoolHoliday', 'StoreType', 'Assortment',
       'CompetitionDistance', 'CompetitionOpenSinceMonth',
       'CompetitionOpenSinceYear', 'Promo2', 'Promo2SinceWeek',
       'Promo2SinceYear', 'PromoInterval', 'Year', 'Month', 'Day',
       'WeekOfYear', 'CompetitionOpen', 'Promo2Open', 'IsPromo2Month'],
      dtype='object')

In [95]:
input_cols = ['Store', 'DayOfWeek', 'Promo', 'StateHoliday', 'StoreType', 'Assortment', 'CompetitionDistance', 'CompetitionOpen', 'Day', 'Month', 'Year', 'WeekOfYear', 'Promo2', 'Promo2Open', 'IsPromo2Month']
target_cols = 'Sales'

In [96]:
input_df = merged_df[input_cols].copy()
target_df = merged_df[target_cols].copy()

In [97]:
%%time
test_inputs = merged_test_df[input_cols].copy()

CPU times: total: 46.9 ms
Wall time: 25.5 ms


In [98]:
%%time
numeric_cols = input_df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('DayOfWeek')
cat_cols = input_df.select_dtypes(include=['object', 'string']).columns.tolist()
cat_cols.append('DayOfWeek')

CPU times: total: 46.9 ms
Wall time: 44.4 ms


In [99]:
numeric_cols, cat_cols

(['Store',
  'Promo',
  'CompetitionDistance',
  'CompetitionOpen',
  'Day',
  'Month',
  'Year',
  'WeekOfYear',
  'Promo2',
  'Promo2Open',
  'IsPromo2Month'],
 ['StateHoliday', 'StoreType', 'Assortment', 'DayOfWeek'])

## Imputing missing data

In [100]:
input_df[numeric_cols].isna().sum(), input_df.shape

(Store                     0
 Promo                     0
 CompetitionDistance    2186
 CompetitionOpen           0
 Day                       0
 Month                     0
 Year                      0
 WeekOfYear                0
 Promo2                    0
 Promo2Open                0
 IsPromo2Month             0
 dtype: int64,
 (844392, 15))

In [101]:
max_distance = input_df.CompetitionDistance.max()
max_distance

np.float64(75860.0)

In [102]:
input_df.fillna({'CompetitionDistance':max_distance*2}, inplace=True)
test_inputs.fillna({'CompetitionDistance':max_distance*2}, inplace=True)

In [103]:
input_df.isna().sum()

Store                  0
DayOfWeek              0
Promo                  0
StateHoliday           0
StoreType              0
Assortment             0
CompetitionDistance    0
CompetitionOpen        0
Day                    0
Month                  0
Year                   0
WeekOfYear             0
Promo2                 0
Promo2Open             0
IsPromo2Month          0
dtype: int64

## Scaling the numeic values

In [104]:
from sklearn.preprocessing import MinMaxScaler

In [105]:
scaler = MinMaxScaler().fit(input_df[numeric_cols])

In [106]:
input_df[numeric_cols] = scaler.transform(input_df[numeric_cols])
test_inputs[numeric_cols] = scaler.transform(test_inputs[numeric_cols])

In [107]:
input_df

,Store,DayOfWeek,Promo,StateHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpen,Day,Month,Year,WeekOfYear,Promo2,Promo2Open,IsPromo2Month
0,0.000000,5,1.0,0,c,a,0.008240,0.059163,1.0,0.545455,1.0,0.588235,0.0,0.000000,0.0
1,0.000898,5,1.0,0,a,a,0.003626,0.066378,1.0,0.545455,1.0,0.588235,1.0,0.890710,1.0
2,0.001795,5,1.0,0,a,a,0.093013,0.074315,1.0,0.545455,1.0,0.588235,1.0,0.720856,1.0
3,0.002693,5,1.0,0,c,c,0.003955,0.050505,1.0,0.545455,1.0,0.588235,0.0,0.000000,0.0
4,0.003591,5,1.0,0,a,a,0.197034,0.002165,1.0,0.545455,1.0,0.588235,0.0,0.000000,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1016776,0.611311,2,0.0,a,b,a,0.000857,0.054834,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0
1016827,0.657092,2,0.0,a,b,b,0.005537,0.114719,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0
1016863,0.689408,2,0.0,a,b,b,0.005405,0.000000,0.0,0.000000,0.0,0.000000,1.0,0.016849,1.0
1017042,0.850090,2,0.0,a,b,b,0.009295,0.000000,0.0,0.000000,0.0,0.000000,0.0,0.000000,0.0


In [108]:
from sklearn.preprocessing import OneHotEncoder

In [109]:
encoder = OneHotEncoder(sparse_output=False, handle_unknown='ignore').fit(input_df[cat_cols])

In [110]:
encoded_cols = encoder.get_feature_names_out().tolist()
encoded_cols

['StateHoliday_0',
 'StateHoliday_a',
 'StateHoliday_b',
 'StateHoliday_c',
 'StoreType_a',
 'StoreType_b',
 'StoreType_c',
 'StoreType_d',
 'Assortment_a',
 'Assortment_b',
 'Assortment_c',
 'DayOfWeek_1',
 'DayOfWeek_2',
 'DayOfWeek_3',
 'DayOfWeek_4',
 'DayOfWeek_5',
 'DayOfWeek_6',
 'DayOfWeek_7']

In [111]:
input_df[encoded_cols] = encoder.transform(input_df[cat_cols])
test_inputs[encoded_cols] = encoder.transform(test_inputs[cat_cols])

In [112]:
input_df.head()

,Store,DayOfWeek,Promo,StateHoliday,StoreType,Assortment,CompetitionDistance,CompetitionOpen,Day,Month,...,Assortment_a,Assortment_b,Assortment_c,DayOfWeek_1,DayOfWeek_2,DayOfWeek_3,DayOfWeek_4,DayOfWeek_5,DayOfWeek_6,DayOfWeek_7
0,0.000000,5,1.0,0,c,a,0.008240,0.059163,1.0,0.545455,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
1,0.000898,5,1.0,0,a,a,0.003626,0.066378,1.0,0.545455,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,0.001795,5,1.0,0,a,a,0.093013,0.074315,1.0,0.545455,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
3,0.002693,5,1.0,0,c,c,0.003955,0.050505,1.0,0.545455,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
4,0.003591,5,1.0,0,a,a,0.197034,0.002165,1.0,0.545455,...,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


## Train/Test Dataframes

In [113]:
print(type(numeric_cols))
print(type(encoded_cols))

<class 'list'>
<class 'list'>


In [114]:
input_df = input_df[numeric_cols + encoded_cols]

In [115]:
split_size = int(len(input_df)*0.8)
split_size

675513

In [116]:
X_train, X_val = input_df[numeric_cols + encoded_cols][:split_size], input_df[numeric_cols + encoded_cols][split_size:]
train_target, val_target = target_df[:split_size], target_df[split_size:]
X_test = test_inputs[numeric_cols + encoded_cols]

# Training the (GBM) Model

We're now ready to train our gradient boosting machine (GBM) model. Here's how a GBM model works:

1. The average value of the target column and uses as an initial prediction every input.
2. The residuals (difference) of the predictions with the targets are computed.
3. A decision tree of limited depth is trained to predict just the residuals for each input.
4. Predictions from the decision tree are scaled using a parameter called the learning rate (this prevents overfitting)
5. Scaled predictions fro the tree are added to the previous predictions to obtain the new and improved predictions.
6. Steps 2 to 5 are repeated to create new decision trees, each of which is trained to predict just the residuals from the previous prediction.

The term "gradient" refers to the fact that each decision tree is trained with the purpose of reducing the loss from the previous iteration (similar to gradient descent). The term "boosting" refers the general technique of training new models to improve the results of an existing model.

EXERCISE: Can you describe in your own words how a gradient boosting machine is different from a random
forest?

# Gradient Boosting - Scikit Learn

---

## 1. Definition

**Gradient Boosting** is an ensemble machine learning technique that builds models sequentially, where each new model tries to correct the errors made by the previous models using gradient descent to minimize a loss function.

It is mainly used for:

* Regression
* Classification

In sklearn it is available as:

* `GradientBoostingRegressor`
* `GradientBoostingClassifier`

---

## 2. How It Works (Step-by-Step)

1. Start with a simple prediction (usually mean of target).
2. Calculate errors (residuals).
3. Train a small decision tree on those errors.
4. Add this tree’s prediction to previous prediction.
5. Repeat this process many times.
6. Final prediction = sum of all weak learners.

It improves gradually by minimizing loss using gradients.

---

## 3. Important Parameters (Sklearn GradientBoosting)

Below are the main parameters with their meaning and typical values.

| Parameter             | Meaning                                     | Common Values                                                |
| --------------------- | ------------------------------------------- | ------------------------------------------------------------ |
| `n_estimators`        | Number of boosting stages (trees)           | 100–1000                                                     |
| `learning_rate`       | Shrinks contribution of each tree           | 0.01–0.3                                                     |
| `max_depth`           | Depth of each tree                          | 3–8                                                          |
| `min_samples_split`   | Minimum samples required to split a node    | 2–20                                                         |
| `min_samples_leaf`    | Minimum samples required at leaf node       | 1–10                                                         |
| `max_features`        | Number of features considered at each split | 'sqrt', 'log2', int                                          |
| `subsample`           | Fraction of samples used for each tree      | 0.5–1.0                                                      |
| `loss`                | Loss function to optimize                   | 'squared_error', 'absolute_error', 'log_loss', 'exponential' |
| `random_state`        | Controls randomness                         | Any integer                                                  |
| `criterion`           | Function to measure split quality           | 'friedman_mse', 'squared_error'                              |
| `validation_fraction` | Fraction used for early stopping            | 0.1–0.2                                                      |
| `n_iter_no_change`    | Stops if no improvement                     | 5–20                                                         |
| `tol`                 | Minimum improvement threshold               | 1e-4                                                         |
| `verbose`             | Print training progress                     | 0, 1, 2                                                      |

---

## 4. Most Important Parameters (Practically)

In real-world tuning, focus mainly on:

* `n_estimators`
* `learning_rate`
* `max_depth`
* `subsample`
* `min_samples_leaf`

---

## 5. Key Trade-off

There is a strong relation between:

```
learning_rate  ↓  → need  n_estimators ↑
learning_rate  ↑  → need  n_estimators ↓
```

Small learning rate + many trees = better performance but slower training.

---

## 6. When To Use Gradient Boosting

Use when:

* You want high accuracy
* Dataset size is medium
* You can afford slower training
* Relationships are complex and nonlinear

Avoid when:

* Dataset is extremely large (use XGBoost/LightGBM instead)
* Real-time training is required

---

## 7. Difference from Random Forest

| Random Forest             | Gradient Boosting        |
| ------------------------- | ------------------------ |
| Trees built independently | Trees built sequentially |
| Reduces variance          | Reduces bias             |
| Faster training           | Slower training          |
| Less prone to overfitting | Can overfit if not tuned |

---

In [117]:
from xgboost import XGBRegressor

# XGBoost – `XGBRegressor`

---

## 1. Definition

**XGBoost (Extreme Gradient Boosting)** is an optimized and regularized implementation of Gradient Boosting that is faster, more accurate, and better at preventing overfitting.

It improves traditional Gradient Boosting by:

* Using second-order gradients
* Adding regularization
* Supporting parallel processing
* Handling missing values automatically

Used mainly for:

* Regression
* Classification
* Ranking problems

---

## 2. How It Works (Conceptually)

1. Start with an initial prediction (usually mean).
2. Compute residuals.
3. Build a tree to predict residuals.
4. Update prediction.
5. Apply regularization to control complexity.
6. Repeat for many boosting rounds.

Final prediction = sum of all boosted trees.

---

## 3. Important Parameters of `XGBRegressor`

Below are the main parameters grouped logically.

---

## 🔹 General Parameters

| Parameter             | Meaning                   | Common Values      |
| --------------------- | ------------------------- | ------------------ |
| `n_estimators`        | Number of boosting rounds | 100–1000           |
| `learning_rate` (eta) | Step size shrinkage       | 0.01–0.3           |
| `max_depth`           | Maximum tree depth        | 3–10               |
| `verbosity`           | Print messages            | 0–3                |
| `n_jobs`              | CPU cores used            | -1 or specific int |
| `random_state`        | Controls randomness       | Any integer        |

---

## 🔹 Tree Structure Parameters

| Parameter           | Meaning                                  | Common Values |
| ------------------- | ---------------------------------------- | ------------- |
| `min_child_weight`  | Minimum sum of weights in a child        | 1–10          |
| `gamma`             | Minimum loss reduction required to split | 0–5           |
| `subsample`         | Fraction of training rows used per tree  | 0.5–1.0       |
| `colsample_bytree`  | Fraction of features per tree            | 0.5–1.0       |
| `colsample_bylevel` | Fraction per tree level                  | 0.5–1.0       |
| `colsample_bynode`  | Fraction per split                       | 0.5–1.0       |

---

## 🔹 Regularization Parameters (Very Important)

| Parameter         | Meaning                         | Common Values |
| ----------------- | ------------------------------- | ------------- |
| `reg_alpha` (L1)  | L1 regularization               | 0–10          |
| `reg_lambda` (L2) | L2 regularization               | 0–10          |
| `max_delta_step`  | Step size control (rarely used) | 0–10          |

---

## 🔹 Boosting Type Parameters

| Parameter     | Meaning                 | Possible Values                              |
| ------------- | ----------------------- | -------------------------------------------- |
| `booster`     | Type of booster         | 'gbtree', 'gblinear', 'dart'                 |
| `tree_method` | Tree building algorithm | 'auto', 'exact', 'hist', 'gpu_hist'          |
| `objective`   | Loss function           | 'reg:squarederror', 'reg:absoluteerror', etc |

---

## 🔹 Early Stopping

| Parameter               | Meaning                                   |
| ----------------------- | ----------------------------------------- |
| `early_stopping_rounds` | Stops if validation score doesn't improve |
| `eval_metric`           | Metric for validation                     |

---

## 4. Most Important Parameters (Practically)

Focus mainly on:

* `n_estimators`
* `learning_rate`
* `max_depth`
* `subsample`
* `colsample_bytree`
* `reg_alpha`
* `reg_lambda`
* `gamma`

These control performance + overfitting.

---

## 5. Why XGBoost is Powerful

Compared to sklearn GradientBoosting:

| Feature             | GradientBoosting | XGBoost     |
| ------------------- | ---------------- | ----------- |
| Speed               | Slower           | Much Faster |
| Parallelization     | Limited          | Yes         |
| Regularization      | Basic            | Advanced    |
| Missing values      | Manual handling  | Automatic   |
| Overfitting control | Moderate         | Strong      |

---

## 6. Bias-Variance Control Strategy

To reduce overfitting:

* Reduce `max_depth`
* Increase `reg_alpha`, `reg_lambda`
* Reduce `subsample`
* Reduce `colsample_bytree`
* Increase `gamma`

To increase model complexity:

* Increase `max_depth`
* Increase `n_estimators`
* Reduce regularization

---

## 7. Typical Safe Starting Configuration

```python
XGBRegressor(
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0,
    reg_lambda=1,
    random_state=42,
    n_jobs=-1
)
```

---

## 8. When To Use XGBoost

Use when:

* Tabular structured data
* Medium to large dataset
* Need high accuracy
* Kaggle competitions
* Complex nonlinear patterns

Avoid when:

* Very small dataset
* Interpretability is critical
* Deep learning is more suitable (images/text)

---

In [118]:
from sklearn.metrics import mean_absolute_error, root_mean_squared_error

def error(true_targets, preds):
    return {'Mean Absolute Error': mean_absolute_error(true_targets, preds),
            'Root Mean Squared Error': root_mean_squared_error(true_targets, preds)}

In [119]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     verbose=1,
                     n_estimators=10,
                     max_depth=5)

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

c:\Users\itspr\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [18:05:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


{'Mean Absolute Error': 1779.9007568359375, 'Root Mean Squared Error': 2421.392333984375}
{'Mean Absolute Error': 1861.6551513671875, 'Root Mean Squared Error': 2489.24658203125}
CPU times: total: 7.31 s
Wall time: 812 ms


In [120]:
# {'train_MAE': 311.5018979475749,
# 'train_rmse': 512.7453573969325,

#  'val_MAE': 896.6415663468015
#  'val_rmse': 1379.1611840541002}

# got these with hyperparametring max_depth=35, n_estimators=50

In [ ]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     n_estimators=50,
                     max_depth=10)

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

c:\Users\itspr\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\training.py:199: UserWarning: [18:05:25] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\learner.cc:790: 
Parameters: { "verbose" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


{'Mean Absolute Error': 605.531494140625, 'Root Mean Squared Error': 853.5949096679688}
{'Mean Absolute Error': 949.282958984375, 'Root Mean Squared Error': 1325.5186767578125}
CPU times: total: 30.5 s
Wall time: 3.39 s


In [122]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     n_estimators=100,
                     max_depth=10)

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

{'Mean Absolute Error': 472.6997375488281, 'Root Mean Squared Error': 666.9273071289062}
{'Mean Absolute Error': 877.616455078125, 'Root Mean Squared Error': 1237.23388671875}
CPU times: total: 57.6 s
Wall time: 5.89 s


In [123]:
model = XGBRegressor(objective='reg:squarederror',
                     n_estimators=100,
                     max_depth=10,
                     learning_rate=0.5,
                     subsample=0.8,
                     colsample_bytree=0.8,
                     reg_alpha=0.8,
                     reg_lambda=1.0,
                     random_state=42)

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

{'Mean Absolute Error': 450.4894104003906, 'Root Mean Squared Error': 627.5081787109375}
{'Mean Absolute Error': 926.1575317382812, 'Root Mean Squared Error': 1272.851318359375}


In [125]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     n_estimators=200,
                     max_depth=10,
                     learning_rate=0.2,
                     booster='gbtree',
                     objective='reg:absoluteerror')

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

{'Mean Absolute Error': 457.94189453125, 'Root Mean Squared Error': 735.2371826171875}
{'Mean Absolute Error': 843.2518920898438, 'Root Mean Squared Error': 1195.3084716796875}
CPU times: total: 6min 53s
Wall time: 56.5 s


In [132]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     n_estimators=1000,
                     max_depth=10,
                     learning_rate=0.2,
                     booster='gbtree',
                     objective='reg:absoluteerror')

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

{'Mean Absolute Error': 400.7898864746094, 'Root Mean Squared Error': 657.716796875}
{'Mean Absolute Error': 822.6166381835938, 'Root Mean Squared Error': 1164.625}
CPU times: total: 31min 30s
Wall time: 5min 2s


In [ ]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     n_estimators=100,
                     max_depth=5,
                     learning_rate=0.2,
                     booster='dart',
                     objective='reg:squarederror')

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

CPU times: total: 7min 58s
Wall time: 44.2 s


KeyboardInterrupt: 

In [68]:
%%time
model = XGBRegressor(random_state=54,
                     n_jobs=-1,
                     n_estimators=200,
                     max_depth=10)

model.fit(X_train, train_target)
preds_train = model.predict(X_train)
preds_val = model.predict(X_val)
print(error(train_target, preds_train))
print(error(val_target, preds_val))

{'Mean Absolute Error': 387.19989013671875, 'Root Mean Squared Error': 540.0699462890625}
{'Mean Absolute Error': 853.3480834960938, 'Root Mean Squared Error': 1216.541748046875}
CPU times: total: 1min 50s
Wall time: 11.1 s


## K Fold Cross Validation

Notice that we didn't create a validation set before training our XGBoost model. We'll use a different validation strategy this time, called K-fold cross validation ([source](https://vitalflux.com/k-fold-cross-validation-python-example/)):

![](https://vitalflux.com/wp-content/uploads/2020/08/Screenshot-2020-08-15-at-11.13.53-AM.png)




##  K-Fold Cross Validation - Theory

**K-Fold Cross Validation** is a model evaluation technique where the dataset is divided into **K equal parts (folds)**.
The model is trained K times, each time using a different fold as validation data and the remaining folds as training data.

It gives a more reliable estimate of model performance.

---

# 🎯 Why Do We Use It?

If you split data only once (train/test split):

* The result depends heavily on that split.
* You might get lucky or unlucky.
* Model evaluation becomes unstable.

K-Fold solves this by:

* Using the entire dataset for both training and validation.
* Reducing variance in performance estimation.
* Giving a more robust evaluation score.

---

# 🔁 How It Works (Step-by-Step)

Assume:

```
K = 5
```

1. Split data into 5 equal folds.
2. First iteration:

   * Fold 1 → Validation
   * Fold 2–5 → Training
3. Second iteration:

   * Fold 2 → Validation
   * Fold 1,3,4,5 → Training
4. Continue until all folds have been used as validation once.
5. Average the performance scores.

---

# 📊 Example

If validation RMSE scores are:

```
Fold 1 → 620
Fold 2 → 605
Fold 3 → 630
Fold 4 → 615
Fold 5 → 610
```

Final score:

```
Average RMSE = (620 + 605 + 630 + 615 + 610) / 5
```

This average is more reliable than a single split.

---

# 🧠 Why It Is Better

| Normal Train/Test Split | K-Fold               |
| ----------------------- | -------------------- |
| One validation result   | K validation results |
| High variance           | Lower variance       |
| Less reliable           | More stable          |
| Might waste data        | Uses all data        |

---

# 🛠 How It Is Done in Sklearn

```python
from sklearn.model_selection import KFold, cross_val_score

kf = KFold(n_splits=5, shuffle=True, random_state=42)

scores = cross_val_score(model, X, y, cv=kf, scoring='neg_root_mean_squared_error')

print(scores)
print("Average Score:", scores.mean())
```

---

# ⚠ Important Variants

### 1️⃣ Stratified K-Fold

Used for classification when classes are imbalanced.
Maintains class distribution in each fold.

```python
from sklearn.model_selection import StratifiedKFold
```

---

### 2️⃣ TimeSeriesSplit

Used for time-series data (like your Rossmann dataset).
It prevents future data leaking into past training.

---

# 📌 When NOT to Use Regular K-Fold

Do NOT use normal K-Fold when:

* Data is time-based (use TimeSeriesSplit).
* Data has grouped structure (use GroupKFold).

---

# 🔍 Summary

K-Fold Cross Validation:

* Splits data into K parts.
* Trains K times.
* Validates on different fold each time.
* Averages performance.
* Gives more reliable model evaluation.

---


In [53]:
from sklearn.model_selection import KFold

In [135]:
#train_and_evalute function will take the train and validation dataframes and return the model and errors(rmse)
def train_and_evaluate(X_train, train_targets, X_val, val_targets, **params):
    model = XGBRegressor(random_state=54, n_jobs=-1, **params)
    model.fit(X_train, train_targets)
    train_rmse = root_mean_squared_error(model.predict(X_train, train_targets))
    val_rmse = root_mean_squared_error(model.predict(X_val, val_targets))
    return model, train_rmse, val_rmse

In [136]:
#now we can use the KFold utility to create the different training/validations spilts and train a separate model for each fold
def test_parms_kfold(n_splits, **params):
    train_rmses, val_rmses, models = [], [], []
    kfold = KFold(n_splits)
    for train_idxs, val_idxs in kfold.split(input_df):
        X_train, train_targets = input_df.iloc[train_idxs], target_df.iloc[train_idxs]
        X_val, val_targets = input_df.iloc[val_idxs], target_df.iloc[val_idxs]
        model, train_rmse, val_rmse = train_and_evaluate(X_train, train_targets, X_val, val_targets, **params)
        models.append(model)
        train_rmses.append(train_rmse)
        val_rmses.append(val_rmse)
    print('Train Rmse: {}, Validation RMSE: {}'.format(np.mean(train_rmses), np.mean(val_rmses)))
    return models
        
        
# def test_params_kfold(n_splits, **params):
#     train_rmses, val_rmses, models = [], [], []
#     kfold = KFold(n_splits)
#     for train_idxs, val_idxs in kfold.split(X):
#         X_train, train_targets = X.iloc[train_idxs], targets.iloc[train_idxs]
#         X_val, val_targets = X.iloc[val_idxs], targets.iloc[val_idxs]
#         model, train_rmse, val_rmse = train_and_evaluate(X_train, train_targets, X_val, val_targets, **params)
#         models.append(model)
#         train_rmses.append(train_rmse)
#         val_rmses.append(val_rmse)
#     print('Train RMSE: {}, Validation RMSE: {}'.format(np.mean(train_rmses), np.mean(val_rmses)))
#     return models

In [ ]:
kfold = KFold(n_splits=5)